In [ ]:
from arch import arch_model
import numpy as np
import pandas as pd
from scipy.stats import norm

# =========================================
# VaR GARCH(1,1) con distribución Normal
# =========================================
# GARCH(1,1) modela la volatilidad condicional σ²_t como:
#   σ²_t = ω + α·ε²_{t-1} + β·σ²_{t-1}
# donde ε_{t-1} es el shock del día anterior y σ²_{t-1} la varianza pasada.
#
# Distribución de errores: Normal → tiende a subestimar el riesgo en las colas
# (fat tails de los rendimientos financieros no son bien capturadas).
#
# Configuración (igual que MLP para comparación justa):
#   - Ventana de entrenamiento: 900 observaciones
#   - Reentrenamiento: diario (parámetros GARCH reestimados cada sesión)
# =========================================

# Cargamos el dataset preprocesado (generado en datos.ipynb)
dataset = pd.read_pickle("../data/dataset_tfg.pkl").copy().sort_index()

alpha = 0.99       # nivel de confianza del VaR (99%)
retrain_every = 1  # reestimamos los parámetros GARCH cada día

eval_start = pd.Timestamp("2015-01-01")
eval_end   = pd.Timestamp("2024-12-31")
eval_dates = dataset.loc[eval_start:eval_end].index

var_preds = []
real_targets = []
dates = []
current_model = None  # almacena el último modelo GARCH válido estimado

print("Iniciando GARCH-Normal rolling...\n")

for i, current_date in enumerate(eval_dates):

    if i % retrain_every == 0:
        # Seleccionamos las 900 observaciones anteriores al día actual
        train_end = current_date - pd.Timedelta(days=1)
        train_df  = dataset.loc[:train_end].tail(900)

        # La librería arch trabaja con retornos expresados en porcentaje (×100)
        # Usamos rp_lag_0 = rendimiento de la cartera del día anterior
        r_train = (train_df["rp_lag_0"].dropna() * 100).astype(float)

        if len(r_train) >= 250:
            # Especificación: media constante, volatilidad GARCH(1,1), errores normales
            am = arch_model(
                r_train,
                mean="Constant",
                vol="GARCH",
                p=1, q=1,
                dist="normal"
            )
            try:
                # Estimación por máxima verosimilitud; disp="off" suprime output
                current_model = am.fit(disp="off")
            except Exception:
                pass   # si la estimación no converge, mantenemos el último modelo válido

    if current_model is None:
        continue

    try:
        # Predicción de la varianza condicional para el siguiente día (horizonte h=1)
        forecast = current_model.forecast(horizon=1, reindex=False)
        mu_t    = forecast.mean.iloc[-1, 0]               # media condicional estimada
        sigma_t = np.sqrt(forecast.variance.iloc[-1, 0])  # desv. típica condicional

        # Cuantil de la Normal al nivel (1 - alpha) = 1% → valor negativo (cola izquierda)
        z = norm.ppf(1 - alpha)

        # VaR como retorno: μ + σ·z  (en porcentaje, con z < 0)
        # Convertimos a pérdida positiva: negamos y dividimos por 100
        var_return = mu_t + sigma_t * z
        var_loss_pred = -var_return / 100.0

        real_loss = dataset.loc[current_date, "target"]

        var_preds.append(var_loss_pred)
        real_targets.append(real_loss)
        dates.append(current_date)

    except Exception:
        continue  # si la predicción falla, omitimos ese día

# ==============================
# Resultados y métricas básicas
# ==============================
out_garch_n = pd.DataFrame(
    {"VaR_GARCH_n": np.array(var_preds), "loss_real": np.array(real_targets)},
    index=pd.to_datetime(dates)
).sort_index()

out_garch_n = out_garch_n.loc["2015":"2024"].dropna().copy()

# Indicador de violación: 1 si la pérdida real supera el VaR estimado
out_garch_n["viol"] = (out_garch_n["loss_real"] > out_garch_n["VaR_GARCH_n"]).astype(int)
out_garch_n["year"] = out_garch_n.index.year

# Tasa de violación: una tasa > 1% indica subestimación del riesgo
viol_rate = out_garch_n["viol"].mean()
yearly = out_garch_n.groupby("year")["viol"].mean()

print("\n===================================")
print("VaR GARCH(1,1) Normal")
print("===================================")
print(f"Ventana         : 900 obs")
print(f"Retrain cada    : {retrain_every} día(s)")
print(f"Total preds     : {len(out_garch_n)}")
print(f"Violation rate  : {viol_rate:.4f}")
print(f"Esperado        : {1-alpha:.4f}")

print("\nViolaciones por año:")
for y, v in yearly.items():
    print(f"{y}: {v:.4f}")

# Exportamos para el análisis comparativo en comp_final.ipynb
out_garch_n.to_csv("../data/garch_n_var_predictions.csv")
print("\nGuardado: garch_n_var_predictions.csv")